## Running LYNX

In [ ]:
import os
import gc
import sys
import time

import pickle
import gzip
import tifffile

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import scFates as scf

import torch
import torch.nn as nn
import torch.nn.functional as F
import pyro

import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

from torch.utils.data import random_split
from torch_geometric.loader import DataLoader

import IO, utils, plot, configs, dataset, trajectory
from models import vgae, model_train

### VGAE (Xenium-DESI integration)

- $z_i \mid u_i \sim \mathcal{N}(f_{\lambda_{\mu}}(u_i), f_{\lambda_{\sigma}}(u_i))$
- $x_i \mid z_i; \mathbf{A} \sim \mathcal{NegBinom}(l \cdot f_{\theta}(z_i; \mathbf{A}), \theta_g)$

In [ ]:
# Load paired Xenium & DESI

xenium_path = '../data/xenium/'
desi_path = '../data/desi/'
metadata_path = '../data/sample_metadata.csv'

n_desi = 100
run_single_sample = True

if run_single_sample:
    sample_id = 'NIH_F5'
    adata = IO.load_multiomics(
        sample_id, xenium_path, desi_path, 
        n_features=n_desi,
        project=True   # Project raw DESI pixel to mapped adata cells as the aux. variable
    )
else:
    # All samples
    sample_ids = sorted([
        f for f in os.listdir(xenium_path) 
        if os.path.isdir(os.path.join(xenium_path, f))
    ])
    
    # metadata_df = pd.read_csv(metadata_path, index_col=[0])
    # metadata_cols = ['SEX', 'BMI']
    # metadata_df = metadata_df[metadata_cols]
    # metadata_df['SEX'] = metadata_df['SEX'].apply(lambda x: 0 if x == 'F' else 1)
    
    adatas = []
    for sample_id in sample_ids:
        adata = IO.load_multiomics(
            sample_id, xenium_path, desi_path,
            n_features=n_desi, 
            # mdata_df = metadata_df
        )
        adatas.append(adata)

gc.collect()


#### Model training: 

In [ ]:
if run_single_sample:
    loader = dataset.XeniumDataset(k=30, n_subgraphs=8)
    graph_data = loader.load_graphs([adata])
    train_data, val_data = random_split(graph_data, [0.8, 0.2])
    train_dl = DataLoader(graph_data, shuffle=True)
    val_dl = DataLoader(graph_data)
    
else:
    # Randomly split samples for training / test (per gender)
    split_ratio = 0.75
    female_indices = np.arange(len(adatas)//2)
    male_indices = np.arange(len(adatas)//2, len(adatas))
    
    train_indices = list(np.random.choice(
        female_indices, 
        size=int(len(female_indices)*split_ratio), replace=False
    )) + list(np.random.choice(
        male_indices, 
        size=int(len(male_indices)*split_ratio), replace=False
    ))

    val_indices = np.setdiff1d(np.arange(len(adatas), train_indices))

    loader = dataset.XeniumDataset(k=30, n_subgraphs=16)
    train_data = loader.load_graphs([adatas[i] for i in train_indices])
    val_data = loader.load_graphs([adatas[i] for i in val_indices])
    
    train_dl = DataLoader(train_data, shuffle=True)
    val_dl = DataLoader(val_data, shuffle=True)

    del female_indices, male_indices
gc.collect()

In [ ]:
from importlib import reload
reload(IO)
reload(utils)
reload(plot)
reload(configs)
reload(dataset)
reload(vgae)
reload(model_train)


In [ ]:
# Debug: single-view VGAE!!!

torch.manual_seed(0)
device = torch.device('cuda')
lr = 1e-3

n_epochs = 400
n_hidden = 64
n_embedding = 8
n_latent = 6
n_covariate = adata.obsm['X_s'].shape[1] if 'X_s' in adata.obsm_keys() \
              else 0

train_configs = configs.set_train_configs(
    n_epochs=n_epochs, lr=lr, gamma=1.,  
    patience=20, device=device,
)

model_configs = configs.set_model_configs(
    c_in=adata.shape[1], c_aux=n_desi, c_covariate=n_covariate, 
    c_hidden=n_hidden, c_latent=n_latent,
    beta=1., k_hop=3, dropout=0.5, act=nn.SiLU(),
) 

In [ ]:
torch.cuda.empty_cache()
pyro.clear_param_store()

model = vgae.VGAE(model_configs, device=train_configs.device)
model, losses, val_losses = model_train.train_vgae(
    model, train_configs,
    dataloader=train_dl,
    val_dataloader=val_dl,
    DEBUG=True
)

In [ ]:
# param_store = pyro.get_param_store()
# # model.save('../results/single_attn.pt')
# model.save(param_store, '../results/multi_attn.pt')

In [ ]:
plt.figure(figsize=(5, 2))
plt.plot(np.arange(len(losses)), losses, label='Train')
plt.plot(np.arange(len(val_losses)), val_losses, label='Val')
plt.legend()
plt.xlabel('Epochs')
plt.ylabel('ELBO')
plt.show()

In [ ]:
def plot_factor_corr(z):
    from scipy.special import comb
    z_corr = np.corrcoef(z.T)
    z_score = np.abs(np.tril(z_corr, k=-1)).sum() / comb(z_corr.shape[0], 2)

    g = sns.clustermap(z_corr, cmap='RdBu_r')
    g.figure.suptitle(
        'q(z)\n Correlation score: {}'.format(np.round(z_score, 3)), 
        fontsize=30, y=1.05
    )
    plt.show()

In [ ]:
# VGAE without conditional 
display_trajectory = True
k = 30
n_zones = 5
n_subgraphs = 1
device = torch.device('cpu')
dist_metric = 'knn'

pyro.clear_param_store()
torch.cuda.empty_cache()

if run_single_sample:
    preds = model.evaluate(adata, k=k, n_subgraphs=n_subgraphs, device=device)
    adata.obsm['X_z'] = preds.qz
    
    if display_trajectory:
        
        # Factor disentanglement
        plot_factor_corr(preds.qz)
            
        # Trajectory inference
        trajectory.compute_trajectory(
            adata, 
            use_rep='X_z',
            n_nodes=n_latent,
            dist_metric=dist_metric,
        )
        
        plot.disp_trajectory(
            adata, 
            cmap='RdBu',
            title='Spatial Gradients\n LYNX'
        )
        
        if 'milestones_colors' in adata.uns_keys():
            adata.uns.pop('milestones_colors')

        sq.pl.spatial_scatter(
            adata, color='t', 
            cmap='RdBu', size=20, img=False,
            title='Pseudotime\n'+'LYNX'
        )
        plt.show()

        utils.get_zonations(adata, n_zones=n_zones, show=True)
        sq.pl.spatial_scatter(
            adata, color='milestones', 
            cmap='RdBu_r', size=20, img=False, 
            title='Zonation\n'+'LYNX'
        )
        plt.show()

gc.collect()


In [ ]:
# VGAE without conditional 
display_trajectory = True
k = 30
n_zones = 5
n_subgraphs = 1
device = torch.device('cpu')
dist_metric = 'knn'

pyro.clear_param_store()
torch.cuda.empty_cache()

if run_single_sample:
    preds = model.evaluate(adata, k=k, n_subgraphs=n_subgraphs, device=device)
    adata.obsm['X_z'] = preds.qz
    
    if display_trajectory:
        
        # Factor disentanglement
        plot_factor_corr(preds.qz)
            
        # Trajectory inference
        trajectory.compute_trajectory(
            adata, 
            use_rep='X_z',
            n_nodes=n_latent,
            dist_metric=dist_metric,
        )
        
        plot.disp_trajectory(
            adata, 
            cmap='RdBu',
            title='Spatial Gradients\n LYNX'
        )
        
        if 'milestones_colors' in adata.uns_keys():
            adata.uns.pop('milestones_colors')

        sq.pl.spatial_scatter(
            adata, color='t', 
            cmap='RdBu', size=20, img=False,
            title='Pseudotime\n'+'LYNX'
        )
        plt.show()

        utils.get_zonations(adata, n_zones=n_zones, show=True)
        sq.pl.spatial_scatter(
            adata, color='milestones', 
            cmap='RdBu_r', size=20, img=False, 
            title='Zonation\n'+'LYNX'
        )
        plt.show()

gc.collect()


In [ ]:
sq.pl.spatial_scatter(
    adata, color='GPX2', cmap='Blues', size=20, img=False
)

sq.pl.spatial_scatter(
    adata, color='DPT', cmap='Blues', size=20, img=False
)

In [ ]:
x_true = adata.X.A.flatten()
x_reconst = preds.px.flatten()

plt.figure(figsize=(4, 4))
plt.scatter(x_true, x_reconst, s=.5, alpha=.5)
plt.xlabel('x_true')
plt.ylabel('x_reconst')
plt.show()

In [ ]:
# Plot individual `pz`
# pz = preds.pz
# pz_labels = ['z'+str(i) for i in range(pz.shape[1])]
# for i in range(len(pz_labels)):
#     adata.obs[pz_labels[i]] = pz[:, i]

# sq.pl.spatial_scatter(
#     adata,
#     color=pz_labels,
#     img=False, size=20, ncols=4,
#     cmap='Reds'
# )
# plt.show()

# Plot individual `qz`
qz = preds.qz
qz_labels = ['z'+str(i) for i in range(qz.shape[1])]
for i in range(len(qz_labels)):
    adata.obs[qz_labels[i]] = qz[:, i]

sq.pl.spatial_scatter(
    adata,
    color=qz_labels,
    img=False, size=20, ncols=4,
    cmap='Reds'
)
plt.show()

In [ ]:
np.save('../results/lynx_{0}_attn.npy'.format(n_latent), adata.obsm['X_z'])

In [ ]:
# torch.save(model.state_dict(), '../results/multi_cat.pt'.format(n_latent)) 
# torch.save(model.state_dict(), '../results/multi_attn.pt')                                    
# del adatas, train_data, train_dl
# gc.collect()

#### All-samples

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/desi/'
sample_ids = sorted([
    sample_id for sample_id in os.listdir(xenium_path)
    if os.path.isdir(os.path.join(xenium_path, sample_id))
])

In [ ]:
torch.manual_seed(0)
device = torch.device('cuda')
lr = 1e-3
n_epochs = 400

n_obs = 377
n_desi = 100
n_hidden = 64
n_latent = 6
n_zones = 5

n_embedding = 8
n_covariate = 0

k = 30
train_configs = configs.set_train_configs(
    n_epochs=n_epochs, lr=lr, gamma=1.,  
    patience=30, device=device,
)

model_configs = configs.set_model_configs(
    c_in=n_obs, c_aux=n_desi, c_covariate=n_covariate, 
    c_hidden=n_hidden, c_latent=n_latent,
    beta=.5, k_hop=3, dropout=0.5, act=nn.SiLU(),
    
    # Debug NF prior / posterior
    flow=True
) 

In [ ]:
# Training individual samples 

n_subgraphs = 8
n_nodes = 10
dist_metric = 'knn'

for sample_id in sample_ids:
    print('Training on {}...'.format(sample_id))

    # ---------------
    # Loading data
    # ---------------
    adata = IO.load_multiomics(
        sample_id, xenium_path, desi_path, 
        n_features=n_desi,
        project=True
    )

    loader = dataset.XeniumDataset(k=30, n_subgraphs=n_subgraphs)
    graph_data = loader.load_graphs([adata])
    train_data, val_data = random_split(graph_data, [0.8, 0.2])
    train_dl = DataLoader(graph_data, shuffle=True)
    val_dl = DataLoader(graph_data)

    # ---------------
    # Training
    # ---------------
    device = torch.device('cuda')
    # model = vgae.SparseVGAE(model_configs, device=train_configs.device)
    model = vgae.VGAE(model_configs, device=train_configs.device)
    model, losses, val_losses = model_train.train_vgae(
        model, train_configs,
        dataloader=train_dl,
        val_dataloader=val_dl,
        DEBUG=True,
    )
    
    # ---------------------------
    # Inference & visualization
    # ---------------------------
    preds = model.evaluate(adata, k=k, n_subgraphs=1, device=torch.device('cpu'))
    adata.obsm['X_z'] = preds.qz

    del model
    gc.collect()
    torch.cuda.empty_cache()
    pyro.clear_param_store()

    
    # Factor disentanglement
    pz_corr = np.corrcoef(preds.pz.T)
    g = sns.clustermap(pz_corr, cmap='RdBu_r')       
    
    qz_corr = np.corrcoef(adata.obsm['X_z'].T)
    g = sns.clustermap(qz_corr, cmap='RdBu_r')

    # Trajectory inference
    try:
        trajectory.compute_trajectory(
            adata, 
            use_rep='X_z',
            n_nodes=n_nodes,
            dist_metric=dist_metric
        )
        
        plot.disp_trajectory(
            adata, 
            cmap='RdBu',
            title='Spatial Gradients\n LYNX'
        )
        
        if 'milestones_colors' in adata.uns_keys():
            adata.uns.pop('milestones_colors')
            
        sq.pl.spatial_scatter(
            adata, color='t', 
            cmap='RdBu', size=20, img=False,
            title='Pseudotime {}\n'.format(sample_id)+'LYNX'
        )
        plt.show()

        # utils.get_zonations(adata, n_zones=n_zones, show=True)
        # sq.pl.spatial_scatter(
        #     adata, color='milestones', 
        #     cmap='RdBu_r', size=20, img=False, 
        #     title='Zonation\n'+'LYNX'
        # )
        # plt.show()

        time.sleep(5)
        
        # Saving results
        np.save('../results/lynx_{0}_{1}.npy'.format(n_latent, sample_id), adata.obsm['X_z'])
        del adata
        
    except ValueError:
        continue

    del adata
    gc.collect()
    
    print('=============\n\n')

#### Held-out validation - antibody

In [ ]:
import scFates as scf
import igraph
import json
import squidpy as sq

import warnings
warnings.filterwarnings('ignore')

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
metadata_path = '../data/sample_metadata.csv'

sample_ids = np.array(sorted([
    f for f in os.listdir(xenium_path) 
    if os.path.isdir(os.path.join(xenium_path, f))
]))

# metadata_df = pd.read_csv(metadata_path, index_col=[0])
# metadata_cols = ['SEX', 'BMI']
# metadata_df = metadata_df[metadata_cols]
# metadata_df['SEX'] = metadata_df['SEX'].apply(lambda x: 0 if x == 'F' else 1)

# train_ids = sample_ids[train_indices]
# val_ids = sample_ids[np.setdiff1d(np.arange(len(sample_ids)), train_indices)]


In [ ]:
# Load model
gc.collect()
torch.cuda.empty_cache()
adata = IO.load_xenium(os.path.join(xenium_path, sample_ids[0]))

# n_covariate = metadata_df.shape[-1]
n_covariate = 0

n_desi = 100
n_hidden = 64
n_latent = 4
n_features = adata.shape[1]
device = torch.device('cpu')

model_configs = configs.set_model_configs(
    c_in=adata.shape[1], c_aux=n_desi, c_covariate=n_covariate, 
    c_hidden=n_hidden, c_latent=n_latent,
    beta=1., k_hop=1, dropout=0., act=nn.ReLU(), 

    embed_option='attn',
    c_embedding=n_embedding,
    num_heads=1,
) 
del adata

pyro.clear_param_store()
model = vgae.SparseVGAE(model_configs, device=device)
# model.load_state_dict(torch.load('../results/multi_cat.pt'))
model.load('../results/multi_attn.pt')


In [ ]:
display = True
k = 30
n_subgraphs = 1
dist_metric = 'euclidean'

for i, sample_id in enumerate(sample_ids):
# for i, sample_id in enumerate(val_ids):
    # ----------------------------------------
    print('Predicting trajectory of {}...'.format(sample_id))
    print('Computing latent representation...')

    # Load adata
    adata = IO.load_multiomics(
        sample_id, xenium_path, desi_path,
        n_features=n_desi,
        # mdata_df = metadata_df
    )

    preds = model.evaluate(adata, k=k, n_subgraphs=n_subgraphs, device=device)
    px = preds.px
    qz = preds.qz
    adata.obsm['X_z'] = qz

    gc.collect()    
    torch.cuda.empty_cache()

    pz_corr = np.corrcoef(preds.pz.T)
    g = sns.clustermap(pz_corr, cmap='RdBu_r')       
    
    qz_corr = np.corrcoef(preds.qz.T)
    g = sns.clustermap(qz_corr, cmap='RdBu_r')

    # Trajectory inference on the latent representation
    scf.tl.curve(
        adata, 
        use_rep='X_z',
        Nodes=n_latent, 
        epg_extend_leaves=True,
        ndims_rep=n_latent
    )

    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        dist_metric='euclidean', 
    )

    if display:
        # Reconstruction quality
        x_true = adata.X.A.flatten()
        x_reconst = preds.px.flatten()
        
        plt.figure(figsize=(4, 4))
        plt.scatter(x_true, x_reconst, s=.5, alpha=.5)
        plt.xlabel('x_true')
        plt.ylabel('x_reeconst')
        plt.show()

        # Latent representation display
        sc.pp.neighbors(adata, use_rep='X_z')
        sc.tl.umap(adata)
    
        plot.disp_trajectory(
            adata, 
            cmap='RdBu',
            title='Spatial Gradients\n ({})'.format(sample_id)
        )
        sq.pl.spatial_scatter(
            adata, color='t', 
            cmap='RdBu', size=20, img=False,
            title='Spatial Gradients\n ({})'.format(sample_id)
        )
        
        sq.pl.spatial_scatter(
            adata, color='milestones', 
            cmap='tab20', size=20, img=False,
            title='Zonation\n ({})'.format(sample_id)
        )
    
    # ----------------------------------------
    # print('\t Loading paired Ab images...')
    # filename = '../data/integrated/antibody/{}.ome.tif'.format(sample_id)
    # img = tifffile.imread(filename)[1:]
    
    # fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(10, 3))
    # ax1.imshow(img[0], vmax=0.5, cmap='magma')
    # ax1.set_title('CV')
    # ax1.axis('off')

    # ax2.imshow(img[1], vmax=0.5, cmap='magma')
    # ax2.set_title('PC')
    # ax2.axis('off')

    # ax3.imshow(img[2], vmax=0.5, cmap='magma')
    # ax3.set_title('PP')
    # ax3.axis('off')

    # ax4.imshow(img[3], vmax=0.5, cmap='magma')
    # ax4.set_title('PV')
    # ax4.axis('off')

    # plt.tight_layout()
    # plt.show()

    # Save latent
    np.save('../results/lynx_{0}_{1}.npy'.format(sample_id, n_latent), qz)

    print('====================================\n\n')
    del adata, img, preds
    gc.collect()
    

In [ ]:
# Latent representation
sc.pp.neighbors(adata, use_rep='X_z')
sc.tl.umap(adata)

scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent)))
sc.pl.umap(
    adata, color='t',
    vmin=np.quantile(adata.obs['t'], .1),
    vmax=np.quantile(adata.obs['t'], .9),
    cmap='RdBu'
)

sq.pl.spatial_scatter(
    adata, color='t', 
    cmap='RdBu', size=20, img=False,
    vmin=np.quantile(adata.obs['t'], .1),
    vmax=np.quantile(adata.obs['t'], .9),
    title='Pseudotime'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab20', size=20, img=False,
    title='Zonation'
)

In [ ]:
filename = '../data/integrated/antibody/{}.ome.tif'.format(sample_id)
img = tifffile.imread(filename)[1:]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
ax1.imshow(img[1], cmap='magma')
ax1.set_title('Peri-central')
ax1.axis('off')

ax2.imshow(img[2], cmap='magma')
ax2.set_title('Peri-portal')
ax2.axis('off')

plt.tight_layout()
plt.show()


---